# Oferta de caña y precio mundial: Guatemala y sus pares

**Pregunta del inciso III.4.** ¿Responden el área cosechada o la producción de distintos países productores a los cambios del precio y con qué rezago se asocian?

Se fijan antes del cálculo Guatemala y los cinco pares definidos en la segmentación de III.1: Indonesia, Bolivia (Plurinational State of), Viet Nam, Nepal y Kenya. World se presenta como contexto agregado. FAOSTAT informa área cosechada, no sembrada. La serie no mide precio neto esperado, margen ni la información que cada productor tenía al decidir la siembra.

Las ecuaciones se estiman por país con el mismo calendario de 2004–2023 y no se apilan como un panel causal: todos reciben el mismo precio mundial. Cambios y niveles, rezagos 0/1/2, modelos descriptivos fijos, bloques, omisiones de un año y sensibilidades de precio permiten valorar heterogeneidad y estabilidad. Los códigos de calidad quedan visibles y marcados para revisión, sin excluir por el flag solamente.

In [1]:
from pathlib import Path
import sys

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from scripts.analizar_oferta_paises_serie3 import ejecutar

resumen = ejecutar(RAIZ)
resumen

{'huellas': {'data\\raw\\serie3\\Sugar_data.csv': '944471f40adce9a236bc992e3824f8c07f8f87b0f1aa7a5e50c47c888dcd52d1',
  'outputs\\serie3_eda\\revision\\cana_todas_celdas.csv': '8fafc6a6ea00522a25d1ee6aab5c2869c9ee4d0b51b3913efcbe388298bcb36d'},
 'celdas_crudas_totales': 27648,
 'celdas_paises_seleccionados': 1344,
 'celdas_fuera_analisis': 27228,
 'celdas_en_muestra_resultado': 420,
 'filas_panel': 161,
 'paises_comparados': ['Guatemala',
  'Indonesia',
  'Bolivia (Plurinational State of)',
  'Viet Nam',
  'Nepal',
  'Kenya'],
 'World_solo_contexto': True,
 'n_por_pais_objetivo': 20,
 'flags_distintos_de_A_solo_marcados_1961_2023': 287,
 'flags_distintos_de_A_en_muestra_resultado': 54,
 'exclusiones_por_flag': 0,
 'residuo_identidad_log_maximo': 7.951713691056739e-06,
 'n_correlaciones': 126,
 'n_ajustes': 112,
 'n_omisiones': 1120,
 'estimacion_causal': False,
 'disponibilidad_historica_faostat_certificada': False}

## Cobertura, conservación y calidad

La primera tabla verifica que cada variable por área conserve su flag y unidad. Las celdas originales y las apartadas por ámbito/historia quedan en CSV con motivos. E/I/X son marcas de fuente preservadas para inspección; su presencia no se interpreta automáticamente como error.

In [2]:
salida = RAIZ / "outputs/serie3_oferta_paises"
conteos_flags = pd.read_csv(salida / "conteo_flags.csv")
conteos_flags.pivot_table(index=["area", "elemento"], columns="flag", values="n", fill_value=0)

flag                                                A     E    I     X
area                             elemento                             
Bolivia (Plurinational State of) Area harvested  64.0   0.0  0.0   0.0
                                 Production      64.0   0.0  0.0   0.0
                                 Yield           64.0   0.0  0.0   0.0
Guatemala                        Area harvested  57.0   2.0  4.0   1.0
                                 Production      50.0  12.0  1.0   1.0
                                 Yield           44.0  20.0  0.0   0.0
Indonesia                        Area harvested  51.0  11.0  0.0   2.0
                                 Production      14.0   7.0  0.0  43.0
                                 Yield           10.0  52.0  0.0   2.0
Kenya                            Area harvested  50.0   3.0  0.0  11.0
                                 Production      27.0  37.0  0.0   0.0
                                 Yield           27.0  37.0  0.0   0.0
Nepal                            Area harvested  64.0   0.0  0.0   0.0
                                 Production      63.0   1.0  0.0   0.0
                                 Yield           63.0   1.0  0.0   0.0
Viet Nam                         Area harvested  55.0   0.0  0.0   9.0
                                 Production      55.0   0.0  0.0   9.0
                                 Yield           54.0   2.0  0.0   8.0
World                            Area harvested  60.0   4.0  0.0   0.0
                                 Production      61.0   3.0  0.0   0.0
                                 Yield           60.0   4.0  0.0   0.0

## Señales descriptivas en la misma muestra

Las correlaciones de cambios y niveles usan los mismos veinte años. Los modelos son descriptivos y sirven para comparar si la asociación con precio mejora la referencia propia; la evaluación 2014–2023 usa parámetros estimados con años previos, pero precio y FAOSTAT en versión final no certifican disponibilidad histórica ni desempeño operativo.

In [3]:
cor = pd.read_csv(salida / "correlaciones.csv")
corr_cambios = cor[(cor.escala == "g") & cor.objetivo.isin(["superficie", "produccion"])]
corr_cambios.pivot_table(index=["area", "objetivo"], columns="rezago_precio", values="correlacion")

rezago_precio                                       0         1         2
area                             objetivo                                
Bolivia (Plurinational State of) produccion -0.050294  0.029322  0.020149
                                 superficie -0.163115 -0.215531  0.368503
Guatemala                        produccion -0.391043 -0.211253  0.000528
                                 superficie -0.390692  0.222614  0.310510
Indonesia                        produccion  0.171878 -0.113894 -0.195032
                                 superficie  0.071988  0.198184  0.165437
Kenya                            produccion -0.102010 -0.058960 -0.180183
                                 superficie  0.007612  0.181824 -0.149766
Nepal                            produccion  0.121027 -0.217983  0.263868
                                 superficie  0.019488 -0.146142  0.392440
Viet Nam                         produccion -0.116802  0.507164  0.446194
                                 superficie -0.152381  0.526982  0.379840
World                            produccion -0.207432  0.096627  0.775533
                                 superficie -0.181087  0.063415  0.717959

In [4]:
ajustes = pd.read_csv(salida / "ajustes.csv")
metricas = pd.read_csv(salida / "metricas_temporales.csv")
ajustes[(ajustes.modelo == "propia_precio") & (ajustes.bloque == "completo_2004_2023")][
    ["area", "objetivo", "n", "intercepto", "g_previa", "p_1", "p_2", "r2_descriptivo"]
].merge(
    metricas[metricas.modelo == "propia_precio"][["area", "objetivo", "MAE_puntos_log"]],
    on=["area", "objetivo"], validate="one_to_one"
)

,area,objetivo,n,intercepto,g_previa,p_1,p_2,r2_descriptivo,MAE_puntos_log
0,Guatemala,superficie,20,0.005330,0.209993,0.060885,0.049624,0.176925,4.654226
1,Guatemala,produccion,20,0.025000,-0.119295,-0.118559,-0.006435,0.056083,11.224704
2,Indonesia,superficie,20,0.014166,0.196876,0.037752,0.022000,0.101757,5.582193
3,Indonesia,produccion,20,0.017551,-0.209222,-0.019593,-0.058810,0.091400,5.188815
4,Bolivia (Plurinational State of),superficie,20,0.016112,0.274668,-0.058467,0.126513,0.255845,4.707480
5,Bolivia (Plurinational State of),produccion,20,0.042018,-0.161399,0.007370,0.009190,0.027032,8.835163
6,Viet Nam,superficie,20,-0.030189,0.393738,0.210889,0.056706,0.517141,5.886172
7,Viet Nam,produccion,20,-0.028183,0.300717,0.233631,0.117101,0.507099,7.594278
8,Nepal,superficie,20,0.001360,-0.301532,-0.044191,0.098606,0.267774,7.984150
9,Nepal,produccion,20,0.022915,-0.419989,-0.080536,0.080721,0.284750,12.709868


## Lectura gerencial y límites

La comparación responde si la asociación histórica difiere entre Guatemala y pares que ya eran comparables por estructura productiva. Signos parecidos y estables entre bloques/omisiones serían evidencia de consistencia descriptiva, no de causalidad ni de plazo agronómico. Una diferencia aislada tampoco prueba que un país ignore incentivos: expectativas, política, clima, moneda, costos y calendario de cultivo no están observados. No se estima el impacto de una decisión de expansión ni se recomienda una superficie objetivo.

La implicación útil es usar escenarios de precios e incentivos de competidores al evaluar margen esperado y capacidad, y pedir calendario de siembra/renovación, precios netos esperados, costos, rendimientos físicos, clima/agua e historial de versiones FAOSTAT antes de una decisión. No se estima impacto financiero con estos datos.